# SmartSignal-Style Anomaly Detection
Similarity-based anomaly detection that compares current sensor readings against historical normal baselines.
Complements the short-term (4h/8h/24h) and long-term (Cox) predictive models by detecting abnormal sensor behavior in real-time.
**Pipeline:**
1. Build normal operating baselines from 90 days of PI data (running periods, no GADS events).
2. Score current sensor data from Eventhouse against baselines using robust Z-scores.
3. Generate severity-tagged advisories and persist to gold tables, plus a multivariate PCA residual scorer.

In [ ]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timedelta
# Unique run id for traceability of every write
notebook_run_id = str(uuid.uuid4())
# Target assets
TARGET_ASSETS = ['RV2_U2_Boiler', 'RV3_U3_Steam_Turbine', 'RV3_U3_Boiler_Feed_Pump_East']
# Anomaly detection parameters
BASELINE_DAYS = 90          # Days of history to compute baseline
ZSCORE_THRESHOLD = 3.0      # Robust Z threshold for individual anomaly
MULTI_SENSOR_THRESHOLD = 3  # Number of simultaneous anomalous tags to trigger multi-sensor alert
BIN_SECONDS = 900           # 15-minute bins
MW_BIN_WIDTH = 50           # MW load bin width for load-conditioned baselines
EPISODE_GAP_MIN = 60        # Gap in minutes that breaks an episode
STUCK_EPSILON = 0.05        # Fraction of baseline MAD below which sensor is "stuck"
# Running state tags and thresholds (for filtering normal operating periods)
RUNNING_STATE = {
    'RV2_U2_Boiler': {'tag': 'RV2:BTPU2BPDRUM.AG', 'threshold': 600, 'op': '>'},
    'RV3_U3_Steam_Turbine': {'tag': 'RV3:TXSU3TS15A.AG', 'threshold': 6, 'op': '>'},
    'RV3_U3_Boiler_Feed_Pump_East': {'tag': 'RV3:FPSU3TS24E.AG', 'threshold': 20, 'op': '>'}
}
# MW (generator load) tag per asset, used for load conditioning of baselines
MW_TAGS = {
    'RV2_U2_Boiler': 'RV2:GEJU2GE03.AG',
    'RV3_U3_Steam_Turbine': 'RV3:GEJU3GE03.AG',
    'RV3_U3_Boiler_Feed_Pump_East': 'RV3:GEJU3GE03.AG'
}
# Tables
PI_TABLE = "gold.fact_pi"
BRIDGE_TABLE = "gold.bridge_pi_tag_to_asset"
BASELINE_TABLE = "ml.anomaly_baselines"
ADVISORY_TABLE = "ml.anomaly_advisories"
WATCHLIST_TABLE = "ml.watchlist"
MV_SCORE_TABLE = "ml.anomaly_multivariate"
# Eventhouse connection (for real-time scoring)
KUSTO_URI = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
KUSTO_DB = "pi-realtime-db"
def _kusto_tok():
    try:
        import notebookutils as _n; _c = _n.credentials
    except Exception:
        from notebookutils import mssparkutils as _m; _c = _m.credentials
    for _a in (KUSTO_URI, "kusto", "pbi"):
        try:
            _t = _c.getToken(_a)
            if _t: return _t
        except Exception:
            pass
    raise RuntimeError("could not acquire Kusto token")

def read_kusto(query):
    return (spark.read
        .format("com.microsoft.kusto.spark.datasource")
        .option("accessToken", _kusto_tok())
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoQuery", query)
        .load())
def check_running_state():
    """Query Eventhouse for current running state of each asset."""
    kql = """
    let rv2=PiEvents|where not(Questionable) and Tag=="RV2:BTPU2BPDRUM.AG"|top 1 by Ts desc|extend asset_id="RV2_U2_Boiler",Threshold=600;
    let rv3=PiEvents|where not(Questionable) and Tag=="RV3:TXSU3TS15A.AG"|top 1 by Ts desc|extend asset_id="RV3_U3_Steam_Turbine",Threshold=6;
    let bfp=PiEvents|where not(Questionable) and Tag=="RV3:FPSU3TS24E.AG"|top 1 by Ts desc|extend asset_id="RV3_U3_Boiler_Feed_Pump_East",Threshold=20;
    union rv2,rv3,bfp
    |extend V=toreal(Value)
    |extend IsRunning=(V > Threshold)
    |project asset_id, IsRunning, V=round(V,1), Tag, Ts
    """
    try:
        rs_df = read_kusto(kql)
        status = {}
        for row in rs_df.collect():
            status[row.asset_id] = bool(row.IsRunning)
            state = "RUNNING" if row.IsRunning else "STOPPED"
            print(f"  {row.asset_id}: {state} (value={row.V}, tag={row.Tag})")
        for a in TARGET_ASSETS:
            if a not in status:
                status[a] = True
                print(f"  {a}: UNKNOWN ")
        return status
    except Exception as e:
        print(f"  \u26a0\ufe0f Running-state check failed: {e} ")
        return {a: True for a in TARGET_ASSETS}
print("SmartSignal-Style Anomaly Detection initialized")
print(f"  notebook_run_id: {notebook_run_id}")
print(f"  Assets: {len(TARGET_ASSETS)}")
print(f"  Baseline window: {BASELINE_DAYS} days")
print(f"  Robust-Z threshold: {ZSCORE_THRESHOLD}")
print(f"  MW load bin width: {MW_BIN_WIDTH}")
print(f"  Multi-sensor threshold: {MULTI_SENSOR_THRESHOLD}")
print("\n\U0001f50d Checking current asset running state...")
ASSET_RUNNING = check_running_state()
ACTIVE_ASSETS = [a for a in TARGET_ASSETS if ASSET_RUNNING.get(a, True)]
STOPPED_ASSETS = [a for a in TARGET_ASSETS if not ASSET_RUNNING.get(a, True)]
if STOPPED_ASSETS:
    print(f"\n\u26a0\ufe0f Stopped assets (will skip scoring): {', '.join(STOPPED_ASSETS)}")
print(f"\u2713 Active assets for scoring: {', '.join(ACTIVE_ASSETS)}")
# Sanitize PI tag names for use as Spark column names (: and . are reserved)
import re
def safe_col(tag):
    return re.sub(r'[:\.\-]', '_', tag)

## Phase 1: Build Normal Operating Baselines
Compute per-tag statistics (median, MAD, percentiles) from historical data during **normal operating periods only**.
Normal = asset is running AND no active GADS forced outage. Baselines are conditioned on MW load bin (50 MW buckets) so behavior is compared at like-for-like operating conditions.

In [ ]:
print("="*70)
print("PHASE 1: BUILDING NORMAL OPERATING BASELINES")
print("="*70)
# --- Load PI data with asset mapping ---
# gold.fact_pi already carries the canonical friendly asset_id (e.g. RV2_U2_Boiler);
# use it directly. The bridge's asset_id is a numeric equipment id that never matches
# TARGET_ASSETS, which silently dropped every row (0-row baselines).
pi_data = spark.table(PI_TABLE).filter(
    F.col("asset_id").isin(TARGET_ASSETS) &
    F.col("ValueNumeric").isNotNull() &
    (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
).cache()
total_pi = pi_data.count()
print(f"\u2713 Loaded PI data: {total_pi:,} rows ({BASELINE_DAYS}-day window)")
# --- IMPROVEMENT #1: Build running-state windows per asset using RUNNING_STATE ---
# Read each asset's running-state tag, mark "running" samples by threshold/op,
# detect transitions with a Window, and collapse contiguous running samples into
# [run_start, run_end] windows. Inner-join PI data to keep only running periods.
running_windows_list = []
for asset, cfg in RUNNING_STATE.items():
    rs_tag, thr, op = cfg['tag'], cfg['threshold'], cfg['op']
    rs = spark.table(PI_TABLE).filter(
        (F.col("Tag") == rs_tag) &
        F.col("ValueNumeric").isNotNull() &
        (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
    ).select("Timestamp", "ValueNumeric")
    if op == '>':
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") > thr).cast("int"))
    elif op == '>=':
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") >= thr).cast("int"))
    elif op == '<':
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") < thr).cast("int"))
    elif op == '<=':
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") <= thr).cast("int"))
    else:
        rs = rs.withColumn("is_running", (F.col("ValueNumeric") == thr).cast("int"))
    w = Window.orderBy("Timestamp")
    rs = rs.withColumn("prev_state", F.lag("is_running").over(w))
    rs = rs.withColumn("state_change", (F.col("is_running") != F.coalesce(F.col("prev_state"), F.lit(-1))).cast("int"))
    rs = rs.withColumn("grp", F.sum("state_change").over(w.rowsBetween(Window.unboundedPreceding, 0)))
    windows = rs.filter(F.col("is_running") == 1).groupBy("grp").agg(
        F.min("Timestamp").alias("run_start"),
        F.max("Timestamp").alias("run_end")
    ).withColumn("asset_id", F.lit(asset)).select("asset_id", "run_start", "run_end")
    running_windows_list.append(windows)
if running_windows_list:
    running_windows = running_windows_list[0]
    for w_df in running_windows_list[1:]:
        running_windows = running_windows.unionByName(w_df)
    rw_count = running_windows.count()
    print(f"\u2713 Built {rw_count} running-state windows across {len(RUNNING_STATE)} assets")
    pi_data_run = pi_data.alias("p").join(
        F.broadcast(running_windows.alias("rw")),
        (F.col("p.asset_id") == F.col("rw.asset_id")) &
        (F.col("p.Timestamp") >= F.col("rw.run_start")) &
        (F.col("p.Timestamp") <= F.col("rw.run_end")),
        "inner"
    ).select("p.*")
else:
    pi_data_run = pi_data
run_count = pi_data_run.count()
print(f"\u2713 PI data after running-state filter: {run_count:,} rows ({100*run_count/max(total_pi,1):.1f}% of total)")
# --- Anti-join GADS forced-outage windows ---
gads_events = spark.table("gold.fact_gads_event").filter(
    F.col("asset_id").isin(TARGET_ASSETS) &
    F.col("EVENT_TYPE_CD").isin(['U1', 'U2', 'U3', 'MO', 'SF', 'PO', 'RS', 'D1', 'D2', 'D3', 'D4', 'D']) &
    F.col("REAL_START_DT").isNotNull() &
    F.col("REAL_END_DT").isNotNull()
).select(
    F.col("asset_id").alias("event_asset"),
    F.col("REAL_START_DT").alias("event_start"),
    F.col("REAL_END_DT").alias("event_end")
)
normal_pi = pi_data_run.join(
    F.broadcast(gads_events),
    (pi_data_run.asset_id == gads_events.event_asset) &
    (pi_data_run.Timestamp >= gads_events.event_start) &
    (pi_data_run.Timestamp <= gads_events.event_end),
    "left_anti"
).cache()
normal_count = normal_pi.count()
print(f"\u2713 Normal operating data (running AND no GADS event): {normal_count:,} rows ({100*normal_count/max(total_pi,1):.1f}% of total)")
# --- IMPROVEMENT #2: Load conditioning via MW bin per asset ---
# Build a per-asset, per-15min MW value from each asset's MW tag, then attach
# mw_bin (50 MW buckets) to every normal PI reading.
mw_frames = []
for asset, mw_tag in MW_TAGS.items():
    mw_df = spark.table(PI_TABLE).filter(
        (F.col("Tag") == mw_tag) &
        F.col("ValueNumeric").isNotNull() &
        (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
    ).select(
        F.lit(asset).alias("asset_id"),
        F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start").alias("bin_start"),
        F.col("ValueNumeric").alias("mw_value")
    ).groupBy("asset_id", "bin_start").agg(F.avg("mw_value").alias("mw_value"))
    mw_frames.append(mw_df)
mw_per_bin = mw_frames[0]
for f_df in mw_frames[1:]:
    mw_per_bin = mw_per_bin.unionByName(f_df)
mw_per_bin = mw_per_bin.withColumn(
    "mw_bin", (F.floor(F.col("mw_value") / F.lit(MW_BIN_WIDTH)) * F.lit(MW_BIN_WIDTH)).cast("int")
).select("asset_id", "bin_start", "mw_bin")
mw_per_bin.cache()
print(f"\u2713 Computed MW bins: {mw_per_bin.count():,} (asset, 15-min) windows")
normal_pi_binned = normal_pi.withColumn(
    "bin_start", F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start")
).join(mw_per_bin, ["asset_id", "bin_start"], "left").withColumn(
    "mw_bin", F.coalesce(F.col("mw_bin"), F.lit(-1))
)
# --- IMPROVEMENT #2: median + MAD aggregation (robust baseline) ---
# Step 1: Compute per-group median first (avoids Spark nested-aggregate error)
group_medians = normal_pi_binned.groupBy("asset_id", "Tag", "mw_bin").agg(
    F.expr("percentile_approx(ValueNumeric, 0.5, 10000)").alias("group_median")
)
# Step 2: Join median back so abs(value - median) is a plain expression, not a nested agg
normal_with_median = normal_pi_binned.join(group_medians, ["asset_id", "Tag", "mw_bin"], "inner")
pcts_col = F.expr("percentile_approx(ValueNumeric, array(0.05, 0.25, 0.50, 0.75, 0.95), 10000)").alias("pcts")
baselines = normal_with_median.groupBy("asset_id", "Tag", "mw_bin").agg(
    F.count("*").alias("n_readings"),
    pcts_col,
    F.expr("percentile_approx(abs(ValueNumeric - group_median), 0.5, 10000)").alias("baseline_mad_raw"),
    F.min("ValueNumeric").alias("baseline_min"),
    F.max("ValueNumeric").alias("baseline_max"),
    F.min("Timestamp").alias("baseline_start"),
    F.max("Timestamp").alias("baseline_end")
).select(
    "asset_id", "Tag", "mw_bin", "n_readings",
    F.col("pcts")[0].alias("pct_5"),
    F.col("pcts")[1].alias("pct_25"),
    F.col("pcts")[2].alias("pct_50"),
    F.col("pcts")[3].alias("pct_75"),
    F.col("pcts")[4].alias("pct_95"),
    F.col("pcts")[2].alias("baseline_median"),
    # Guard against MAD == 0: floor at 5% of the 5-95 range so near-constant sensors
    # (e.g. fuel oil flow, discrete signals) do not produce extreme Z-scores
    F.greatest(F.col("baseline_mad_raw"), (F.col("pcts")[4] - F.col("pcts")[0]) * F.lit(0.05)).alias("baseline_mad"),
    "baseline_min", "baseline_max", "baseline_start", "baseline_end"
).withColumn("baseline_computed_at", F.current_timestamp()) \
 .withColumn("notebook_run_id", F.lit(notebook_run_id))
# --- IMPROVEMENT #2: Filter baselines: require n>=200 and MAD>0 ---
baselines = baselines.filter((F.col("n_readings") >= 200) & (F.col("baseline_mad") > 0))
bcount = baselines.count()
print(f"\u2713 Computed {bcount} (asset, tag, mw_bin) baselines (median/MAD, load-conditioned)")
print(f"\nBaselines per asset:")
baselines.groupBy("asset_id").agg(
    F.count("*").alias("rows"),
    F.countDistinct("Tag").alias("tags"),
    F.countDistinct("mw_bin").alias("mw_bins"),
    F.avg("n_readings").cast("int").alias("avg_readings")
).show(truncate=False)
baselines.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(BASELINE_TABLE)
print(f"\u2713 Saved baselines to {BASELINE_TABLE}")

## Phase 2: Score Current Sensor Data
Compare the latest sensor readings against baselines. Compute robust Z-scores and flag anomalies.
Uses real-time data from Eventhouse for the most recent readings. Each reading is joined to its (asset, tag, mw_bin) baseline.

In [ ]:
print("="*70)
print("PHASE 2: SCORING CURRENT SENSOR DATA")
print("="*70)
scoring_time = datetime.utcnow()
lookback_hours = 4  # Score the last 4 hours
# Filter to only running assets
scoring_assets = [a for a in TARGET_ASSETS if ASSET_RUNNING.get(a, True)]
stopped = [a for a in TARGET_ASSETS if not ASSET_RUNNING.get(a, True)]
if stopped:
    print(f"\u23f8\ufe0f Skipping stopped assets: {', '.join(stopped)}")
baselines_df = spark.table(BASELINE_TABLE).cache()
print(f"\u2713 Loaded {baselines_df.count()} baselines (median/MAD per mw_bin)")
# Pull recent PI from Eventhouse for ALL needed tags (sensor tags + MW tags for binning)
sensor_tags = [row.Tag for row in baselines_df.select("Tag").distinct().collect()]
all_tags = sorted(set(sensor_tags) | set(MW_TAGS.values()))
kql_tag_filter = "'" + "','".join(all_tags) + "'"
recent_pi_all = read_kusto(f"""
    PiEvents
    | where Ts > ago({lookback_hours}h)
    | where not(Questionable)
    | where Tag in ({kql_tag_filter})
    | summarize Value = avg(toreal(Value)) by bin(Ts, {BIN_SECONDS}s), Tag
    | project Tag, Timestamp = Ts, ValueNumeric = Value
""")
recent_pi_all = recent_pi_all.filter(F.col("ValueNumeric").isNotNull() & ~F.isnan(F.col("ValueNumeric"))).cache()
raw_count = recent_pi_all.count()
print(f"\u2713 Retrieved {raw_count:,} recent readings ({lookback_hours}h window) across {len(all_tags)} tags")
# recent_pi_all comes from the Eventhouse (no asset_id column); build the Tag->asset
# map from fact_pi's friendly asset_id, NOT the bridge's numeric asset_id.
bridge_b = F.broadcast(spark.table(PI_TABLE).select("Tag", "asset_id").where(F.col("asset_id").isin(TARGET_ASSETS)).distinct())
# Compute MW per asset per 15-min bin for the live window
mw_recent_frames = []
for asset, mw_tag in MW_TAGS.items():
    mw_recent_frames.append(
        recent_pi_all.filter(F.col("Tag") == mw_tag).select(
            F.lit(asset).alias("asset_id"),
            F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start").alias("bin_start"),
            F.col("ValueNumeric").alias("mw_value")
        ).groupBy("asset_id", "bin_start").agg(F.avg("mw_value").alias("mw_value"))
    )
mw_recent = mw_recent_frames[0]
for d in mw_recent_frames[1:]:
    mw_recent = mw_recent.unionByName(d)
mw_recent = mw_recent.withColumn(
    "mw_bin", (F.floor(F.col("mw_value") / F.lit(MW_BIN_WIDTH)) * F.lit(MW_BIN_WIDTH)).cast("int")
).select("asset_id", "bin_start", "mw_bin")
# Build sensor recent frame with asset_id, bin_start, mw_bin
recent_pi = recent_pi_all.join(bridge_b, "Tag", "inner").filter(
    F.col("asset_id").isin(scoring_assets)
).withColumn(
    "bin_start", F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start")
).join(mw_recent, ["asset_id", "bin_start"], "left").withColumn(
    "mw_bin", F.coalesce(F.col("mw_bin"), F.lit(-1))
).cache()
recent_count = recent_pi.count()
print(f"  Sensor readings mapped to assets: {recent_count:,}")
if recent_count == 0:
    print("\u26a0\ufe0f No recent data \u2014 check Eventhouse feed")
    scored = None
else:
    # IMPROVEMENT #2: Robust Z = 0.6745 * (x - median) / MAD
    scored = recent_pi.join(
        baselines_df.select(
            "Tag", "asset_id", "mw_bin", "baseline_median", "baseline_mad", "pct_5", "pct_95"
        ),
        ["Tag", "asset_id", "mw_bin"], "inner"
    ).withColumn(
        "z_score", F.lit(0.6745) * (F.col("ValueNumeric") - F.col("baseline_median")) / F.col("baseline_mad")
    ).withColumn(
        "abs_z_score", F.abs(F.col("z_score"))
    ).withColumn(
        "is_anomaly", F.when(F.abs(F.col("z_score")) > ZSCORE_THRESHOLD, 1).otherwise(0)
    ).withColumn(
        "anomaly_direction",
        F.when(F.col("z_score") > ZSCORE_THRESHOLD, "HIGH")
         .when(F.col("z_score") < -ZSCORE_THRESHOLD, "LOW")
         .otherwise("NORMAL")
    ).cache()
    scored_total = scored.count()
    anomaly_count = scored.filter((F.col("is_anomaly") == 1) & F.col("z_score").isNotNull() & ~F.isnan(F.col("z_score"))).count()
    print(f"\n\u2713 Scored {scored_total:,} readings (robust Z, joined on Tag+asset+mw_bin)")
    print(f"  Anomalies detected: {anomaly_count} ({100*anomaly_count/max(scored_total,1):.1f}%)")
    print(f"\nTop anomalies (|robust Z| > {ZSCORE_THRESHOLD}):")
    scored.filter(F.col("is_anomaly") == 1) \
        .orderBy(F.desc("abs_z_score")) \
        .select("asset_id", "Tag", "Timestamp", "mw_bin", "ValueNumeric", "baseline_median", "baseline_mad", "z_score", "anomaly_direction") \
        .show(20, truncate=False)
    # --- Stuck-sensor detection ---
    # If stddev over last 1h < STUCK_EPSILON * baseline_mad, sensor is stuck
    w1h = Window.partitionBy("asset_id", "Tag").orderBy(F.col("Timestamp").cast("long")).rangeBetween(-3600, 0)
    stuck = scored.withColumn("rolling_std_1h", F.stddev("ValueNumeric").over(w1h)) \
        .withColumn("is_stuck",
            (F.col("rolling_std_1h").isNotNull()) &
            (F.col("rolling_std_1h") < F.lit(STUCK_EPSILON) * F.col("baseline_mad"))
        )
    stuck_count = stuck.filter(F.col("is_stuck")).select("asset_id", "Tag").distinct().count()
    print(f"\n\u2713 Stuck-sensor candidates (rolling 1h std < {STUCK_EPSILON} * MAD): {stuck_count}")
    # --- Multi-sensor anomaly grouping using F.window (not raw Timestamp) ---
    multi_sensor = scored.filter((F.col("is_anomaly") == 1) & F.col("z_score").isNotNull() & ~F.isnan(F.col("z_score"))) \
        .groupBy("asset_id", F.window("Timestamp", "15 minutes").alias("win")) \
        .agg(
            F.countDistinct("Tag").alias("anomalous_tags"),
            F.collect_set("Tag").alias("anomalous_tag_list"),
            F.max("abs_z_score").alias("max_z_score")
        ).filter(F.col("anomalous_tags") >= MULTI_SENSOR_THRESHOLD) \
         .select("asset_id", F.col("win.start").alias("window_start"), F.col("win.end").alias("window_end"),
                 "anomalous_tags", "anomalous_tag_list", "max_z_score")
    multi_count = multi_sensor.count()
    if multi_count > 0:
        print(f"\n\ud83d\udd34 MULTI-SENSOR ALERTS: {multi_count} 15-min windows with {MULTI_SENSOR_THRESHOLD}+ simultaneous anomalies")
        multi_sensor.orderBy(F.desc("max_z_score")).show(10, truncate=False)
    else:
        print(f"\n\u2705 No multi-sensor alerts (threshold: {MULTI_SENSOR_THRESHOLD}+ simultaneous)")

## Phase 2b: PCA Multivariate Residual Scorer
For each asset, pivot normal PI data into a wide matrix (one column per tag, rows = 15-min windows), fit StandardScaler + PCA (k=min(10, n_tags)) on the baseline, then compute reconstruction error on recent data as a multivariate anomaly score. Results land in `ml.anomaly_multivariate`.

In [ ]:
print("="*70)
print("PHASE 2b: PCA MULTIVARIATE RESIDUAL SCORER")
print("="*70)
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.feature import PCA as MLPCA
from pyspark.ml import Pipeline
import numpy as np
def pivot_wide(df, tag_list, use_safe_names=False):
    """Pivot tags to columns. If use_safe_names, renames Tag to SafeTag first."""
    if use_safe_names:
        safe_map = {t: safe_col(t) for t in tag_list}
        safe_list = [safe_map[t] for t in tag_list]
        df = df.withColumn("Tag", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("Tag"), ":", "_"), "\\.", "_"), "-", "_"))
        pivot_col_list = safe_list
    else:
        pivot_col_list = tag_list
    return df.withColumn(
        "bin_start", F.window("Timestamp", f"{BIN_SECONDS} seconds").getField("start")
    ).groupBy("bin_start").pivot("Tag", pivot_col_list).agg(F.avg("ValueNumeric"))
mv_results = []
for asset in ACTIVE_ASSETS:
    asset_tags = [r.Tag for r in baselines_df.filter(F.col("asset_id") == asset).select("Tag").distinct().collect()]
    if len(asset_tags) < 3:
        print(f"  [{asset}] skipping \u2014 only {len(asset_tags)} tags with baselines")
        continue
    # Training: normal historical data for this asset
    # Use fact_pi's own friendly asset_id (not the bridge's numeric asset_id).
    train_long = spark.table(PI_TABLE).filter(
        (F.col("asset_id") == asset) &
        F.col("Tag").isin(asset_tags) &
        F.col("ValueNumeric").isNotNull() &
        (F.col("Timestamp") >= F.date_sub(F.current_timestamp(), BASELINE_DAYS))
    )
    safe_tags = [safe_col(t) for t in asset_tags]
    train_wide = pivot_wide(train_long, asset_tags, use_safe_names=True).dropna()
    n_train = train_wide.count()
    if n_train < 50:
        print(f"  [{asset}] skipping \u2014 only {n_train} complete training windows")
        continue
    k = min(10, len(asset_tags))
    assembler = VectorAssembler(inputCols=safe_tags, outputCol="features_raw")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
    pca = MLPCA(k=k, inputCol="features", outputCol="pca_features")
    pipe = Pipeline(stages=[assembler, scaler, pca]).fit(train_wide)
    pca_model = pipe.stages[-1]
    # Reconstruction: x_hat = (x @ V) @ V.T, error = ||x - x_hat||
    V = np.array(pca_model.pc.toArray())  # (d x k)
    V_bcast = spark.sparkContext.broadcast(V)
    def recon_err(scaled_vec):
        x = np.array(scaled_vec.toArray())
        Vm = V_bcast.value
        proj = x @ Vm
        recon = proj @ Vm.T
        return float(np.sqrt(np.sum((x - recon) ** 2)))
    recon_udf = F.udf(recon_err, "double")
    if scored is None:
        continue
    recent_asset = recent_pi.filter(F.col("asset_id") == asset).filter(F.col("Tag").isin(asset_tags))
    recent_wide = pivot_wide(recent_asset, asset_tags, use_safe_names=True).dropna()
    if recent_wide.count() == 0:
        continue
    recent_scored = pipe.transform(recent_wide).withColumn(
        "recon_error", recon_udf(F.col("features"))
    ).select(
        F.lit(asset).alias("asset_id"),
        F.col("bin_start").alias("window_start"),
        F.col("recon_error"),
        F.lit(k).alias("pca_k"),
        F.lit(len(asset_tags)).alias("n_tags"),
        F.lit(notebook_run_id).alias("notebook_run_id"),
        F.current_timestamp().alias("scored_at")
    )
    mv_results.append(recent_scored)
    print(f"  [{asset}] trained PCA k={k} on {n_train} windows; scored {recent_scored.count()} live windows")
if mv_results:
    mv_all = mv_results[0]
    for d in mv_results[1:]:
        mv_all = mv_all.unionByName(d)
    mv_all.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(MV_SCORE_TABLE)
    print(f"\n\u2713 Wrote multivariate scores to {MV_SCORE_TABLE}")
    mv_all.orderBy(F.desc("recon_error")).show(10, truncate=False)
else:
    print("\u26a0\ufe0f No multivariate scores produced")

## Phase 3: Generate Advisories & Persist
Collapse individual anomalous bins into **episodes** (gaps > 60 min start a new episode) with severity based on peak |Z| AND duration. Idempotent MERGE on (asset_id, Tag, first_seen) into `ml.anomaly_advisories`.
Also upserts `ml.watchlist` for unified monitoring.

In [ ]:
print("="*70)
print("PHASE 3: GENERATING ADVISORIES (EPISODE-BASED + MERGE)")
print("="*70)
from delta.tables import DeltaTable
if scored is not None and recent_count > 0:
    anom = scored.filter((F.col("is_anomaly") == 1) & F.col("z_score").isNotNull() & ~F.isnan(F.col("z_score")))
    # --- IMPROVEMENT #3: Episode-based dedup ---
    w_epi = Window.partitionBy("asset_id", "Tag").orderBy("Timestamp")
    anom_e = anom.withColumn(
        "prev_ts", F.lag("Timestamp").over(w_epi)
    ).withColumn(
        "gap_min", (F.col("Timestamp").cast("long") - F.col("prev_ts").cast("long")) / 60.0
    ).withColumn(
        "new_episode", F.when(F.col("prev_ts").isNull() | (F.col("gap_min") > EPISODE_GAP_MIN), 1).otherwise(0)
    ).withColumn(
        "episode_id", F.sum("new_episode").over(w_epi.rowsBetween(Window.unboundedPreceding, 0))
    )
    episodes = anom_e.groupBy("asset_id", "Tag", "episode_id").agg(
        F.min("Timestamp").alias("first_seen"),
        F.max("Timestamp").alias("last_seen"),
        F.max("abs_z_score").alias("peak_abs_z"),
        F.avg("abs_z_score").alias("mean_abs_z"),
        F.count("*").alias("n_breach_bins"),
        F.last("ValueNumeric", ignorenulls=True).alias("latest_value"),
        F.last("baseline_median", ignorenulls=True).alias("baseline_median"),
        F.last("baseline_mad", ignorenulls=True).alias("baseline_mad"),
        F.last("anomaly_direction", ignorenulls=True).alias("anomaly_direction"),
        F.last("mw_bin", ignorenulls=True).alias("mw_bin")
    ).withColumn(
        "duration_h", (F.col("last_seen").cast("long") - F.col("first_seen").cast("long")) / 3600.0
    ).withColumn(
        "severity",
        F.when((F.col("peak_abs_z") > 5.0) & (F.col("duration_h") >= 1.0), "CRITICAL")
         .when((F.col("peak_abs_z") > 4.0) | ((F.col("peak_abs_z") > 3.0) & (F.col("duration_h") >= 4.0)), "HIGH")
         .otherwise("MEDIUM")
    ).withColumn("advisory_message",
        F.concat(
            F.col("Tag"), F.lit(" "), F.col("anomaly_direction"),
            F.lit(" episode: peak|Z|="), F.round(F.col("peak_abs_z"), 2),
            F.lit(", duration_h="), F.round(F.col("duration_h"), 2),
            F.lit(", bins="), F.col("n_breach_bins").cast("string"),
            F.lit(", latest="), F.round(F.col("latest_value"), 2),
            F.lit(" vs median="), F.round(F.col("baseline_median"), 2)
        )
    ).withColumn("scored_at", F.current_timestamp()) \
    .withColumn("notebook_run_id", F.lit(notebook_run_id)) \
    .drop("episode_id")
    episode_count = episodes.count()
    print(f"\u2713 Collapsed individual anomalies into {episode_count} episodes")
    if episode_count > 0:
        print("\nEpisodes by severity:")
        episodes.groupBy("severity").count().orderBy("severity").show()
        print("Episodes by asset:")
        episodes.groupBy("asset_id", "severity").count().orderBy("asset_id", "severity").show(truncate=False)
        # --- IMPROVEMENT #4: idempotent MERGE on (asset_id, Tag, first_seen) ---
        def do_merge(target, episodes, episode_count):
            (target.alias("t").merge(
                episodes.alias("s"),
                "t.asset_id = s.asset_id AND t.Tag = s.Tag AND t.first_seen = s.first_seen"
            ).whenMatchedUpdate(set={
                "last_seen": "s.last_seen",
                "peak_abs_z": "greatest(t.peak_abs_z, s.peak_abs_z)",
                "mean_abs_z": "s.mean_abs_z",
                "n_breach_bins": "s.n_breach_bins",
                "duration_h": "s.duration_h",
                "severity": "s.severity",
                "latest_value": "s.latest_value",
                "advisory_message": "s.advisory_message",
                "scored_at": "s.scored_at",
                "notebook_run_id": "s.notebook_run_id"
            }).whenNotMatchedInsertAll().execute())
            print(f"\u2713 MERGED {episode_count} episodes into {ADVISORY_TABLE}")
        try:
            target = DeltaTable.forName(spark, ADVISORY_TABLE)
            try:
                do_merge(target, episodes, episode_count)
            except Exception as merge_err:
                if "UNRESOLVED" in str(merge_err) or "cannot resolve" in str(merge_err).lower():
                    print(f"  Schema mismatch detected ADVISORY_TABLE")
                    spark.sql(f"DROP TABLE IF EXISTS {ADVISORY_TABLE}")
                    episodes.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(ADVISORY_TABLE)
                    print(f"\u2713 Wrote {episode_count} episodes to {ADVISORY_TABLE} (fresh table)")
                else:
                    raise merge_err
        except Exception as e:
            if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "is not a Delta table" in str(e):
                episodes.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(ADVISORY_TABLE)
                print(f"\u2713 Created {ADVISORY_TABLE} with {episode_count} episodes")
            elif "UNRESOLVED" in str(e) or "cannot resolve" in str(e).lower():
                spark.sql(f"DROP TABLE IF EXISTS {ADVISORY_TABLE}")
                episodes.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(ADVISORY_TABLE)
                print(f"\u2713 Recreated {ADVISORY_TABLE} with {episode_count} episodes")
            else:
                raise
        bridge_desc = spark.table("gold.bridge_pi_tag_to_asset").select("Tag", "tag_description").dropDuplicates(["Tag"]).toPandas()
        tag_desc_map = dict(zip(bridge_desc['Tag'], bridge_desc['tag_description']))
        epi_pdf = episodes.orderBy(F.desc("peak_abs_z")).toPandas()
        wl_rows = []
        for asset_id, grp in epi_pdf.groupby("asset_id"):
            top3 = grp.nlargest(3, "peak_abs_z")
            worst_z = top3["peak_abs_z"].max()
            severity = "CRITICAL" if worst_z > 5.0 and top3.iloc[0]["duration_h"] >= 1.0 else "HIGH" if worst_z > 4.0 else "MEDIUM"
            friendly = [tag_desc_map.get(t, t).title() for t in top3["Tag"]]
            watch_str = ", ".join(dict.fromkeys(friendly))
            top_tag = top3.iloc[0]["Tag"]
            wl_rows.append({
                "model_name": "AnomalyDetection_SmartSignal",
                "scoring_date": str(datetime.now().date()),
                "asset_id": asset_id,
                "feature": "anomaly_summary",
                "tag_name": top_tag,
                "descriptor": tag_desc_map.get(top_tag, top_tag).title(),
                "engineering_units": None,
                "current_value": float(top3.iloc[0]["latest_value"]) if top3.iloc[0]["latest_value"] is not None else None,
                "baseline_mean": float(top3.iloc[0]["baseline_median"]) if top3.iloc[0]["baseline_median"] is not None else None,
                "baseline_std": float(top3.iloc[0]["baseline_mad"]) if top3.iloc[0]["baseline_mad"] is not None else None,
                "normal_range_low": None, "normal_range_high": None,
                "risk_contribution": float(worst_z),
                "trend_direction": None, "trend_slope_per_day": None,
                "recommended_action": severity,
                "recommendation_text": f"Watch {watch_str}. Worst anomaly z={worst_z:.1f}.",
                "watch_horizon_days": 14,
                "model_run_timestamp": datetime.now(),
                "notebook_run_id": notebook_run_id
            })
        import pandas as pd
        wl_sdf = spark.createDataFrame(pd.DataFrame(wl_rows))
        wl_sdf = wl_sdf.withColumn("watch_horizon_days", F.col("watch_horizon_days").cast("int"))
        try:
            from delta.tables import DeltaTable as DT2
            wtarget = DT2.forName(spark, WATCHLIST_TABLE)
            (wtarget.alias("t").merge(
                wl_sdf.alias("s"),
                "t.model_name = s.model_name AND t.asset_id = s.asset_id AND t.feature = s.feature AND t.scoring_date = s.scoring_date"
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        except Exception:
            wl_sdf.write.mode("append").format("delta").saveAsTable(WATCHLIST_TABLE)
        print(f"Saved {len(wl_rows)} watchlist entries")
    else:
        print("No anomaly episodes - all sensors within normal range")
        # Write NORMAL watchlist rows so stale CRITICAL entries get superseded
        normal_rows = []
        for asset_id in TARGET_ASSETS:
            normal_rows.append({
                "model_name": "AnomalyDetection_SmartSignal",
                "scoring_date": str(datetime.now().date()),
                "asset_id": asset_id,
                "feature": "anomaly_summary",
                "tag_name": None,
                "descriptor": "All sensors normal",
                "engineering_units": None,
                "current_value": None,
                "baseline_mean": None,
                "baseline_std": None,
                "normal_range_low": None, "normal_range_high": None,
                "risk_contribution": 0.0,
                "trend_direction": None, "trend_slope_per_day": None,
                "recommended_action": "LOW",
                "recommendation_text": "Anomaly detection found no episodes. All sensors within normal range.",
                "watch_horizon_days": 14,
                "model_run_timestamp": datetime.now(),
                "notebook_run_id": notebook_run_id
            })
        import pandas as pd
        wl_normal = spark.createDataFrame(pd.DataFrame(normal_rows))
        wl_normal = wl_normal.withColumn("watch_horizon_days", F.col("watch_horizon_days").cast("int"))
        wl_normal.write.mode("append").format("delta").saveAsTable(WATCHLIST_TABLE)
        print(f"Wrote {len(normal_rows)} NORMAL watchlist rows")
else:
    print("Skipped - no recent data available")

## Summary & Per-Asset Health Score
Compute a composite health score per asset using `100 * exp(-sum(max(0, |Z| - 3)^2) / N_tags)` tions dominate over many small ones.

In [ ]:
print("="*70)
print("ASSET HEALTH SUMMARY")
print("="*70)
if scored is not None and recent_count > 0:
    # IMPROVED health: 100 * exp(-sum(max(0, |Z| - 3)^2) / N_tags)
    excess = F.greatest(F.col("abs_z_score") - F.lit(ZSCORE_THRESHOLD), F.lit(0.0))
    health = scored.withColumn("excess", excess).groupBy("asset_id").agg(
        F.count("*").alias("total_readings"),
        F.sum("is_anomaly").alias("anomalous_readings"),
        F.countDistinct("Tag").alias("total_tags"),
        F.countDistinct(F.when(F.col("is_anomaly") == 1, F.col("Tag"))).alias("anomalous_tags"),
        F.max("abs_z_score").alias("max_z_score"),
        F.avg("abs_z_score").alias("avg_z_score"),
        F.sum(F.col("excess") * F.col("excess")).alias("sum_excess_sq")
    ).withColumn(
        "health_score",
        F.round(F.lit(100.0) * F.exp(-F.col("sum_excess_sq") / F.greatest(F.col("total_tags"), F.lit(1))), 1)
    ).withColumn(
        "health_status",
        F.when(F.col("health_score") >= 95, "HEALTHY")
         .when(F.col("health_score") >= 85, "WATCH")
         .when(F.col("health_score") >= 70, "DEGRADED")
         .otherwise("CRITICAL")
    ).orderBy("health_score")
    print("\nAsset Health Scores (robust, magnitude-weighted):")
    health.select("asset_id", "health_score", "health_status",
                  "total_tags", "anomalous_tags", "max_z_score", "avg_z_score").show(truncate=False)
    for asset in TARGET_ASSETS:
        anomalous = scored.filter((F.col("asset_id") == asset) & (F.col("is_anomaly") == 1))
        if anomalous.count() > 0:
            print(f"\n\u26a0\ufe0f {asset} \u2014 Anomalous Tags:")
            anomalous.select("Tag", "Timestamp", "mw_bin", "ValueNumeric",
                             "baseline_median", "baseline_mad", "z_score", "anomaly_direction") \
                .orderBy(F.desc("abs_z_score")) \
                .show(10, truncate=False)
    print(f"\n{'='*70}")
    print(f"Scoring complete at {datetime.utcnow()}")
    print(f"  notebook_run_id: {notebook_run_id}")
    print(f"  Total readings scored: {scored.count():,}")
    print(f"  Anomalies (robust |Z| > {ZSCORE_THRESHOLD}): {scored.filter(F.col('is_anomaly')==1).count()}")
    print(f"  Assets scored: {len(TARGET_ASSETS)}")
    print(f"{'='*70}")
else:
    print("No data scored \u2014 check Eventhouse feed")